# Text-to-SQL Agent — Demo Interactivo

Este notebook muestra el agente funcionando paso a paso sobre `samples.tpch` en Databricks Free Edition.

**Requisitos**: tener `.env` configurado con `GOOGLE_API_KEY` y las credenciales de Databricks (ver README.md).

**Que hace este notebook**:
1. Conecta a Databricks y lista las 8 tablas de `samples.tpch`
2. Muestra el schema de 2 tablas (customer + lineitem)
3. Hace 3 preguntas al agente y muestra el SQL generado + resultados + chips de metadata
4. Demuestra multi-turn: la 2da pregunta referencia "those customers" de la 1ra

In [ ]:
# Setup: agregar el path del proyecto y cargar .env
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv()

from app.config import settings
print(f"Model: {settings.gemini_model}")
print(f"Databricks: {settings.databricks_host}")
print(f"Target: {settings.fqdn_target_table}")
print(f"Databricks configured: {settings.is_databricks_configured}")

## 1. Conectar a Databricks y listar tablas

In [ ]:
from app.core.schema import list_tables, describe_table

tables = list_tables(settings.databricks_catalog, settings.databricks_schema)
print(f"Tablas en {settings.databricks_catalog}.{settings.databricks_schema}: {len(tables)}\n")
for t in tables:
    print(f"  - {t}")

## 2. Schema de 2 tablas (para entender que columnas hay)

In [ ]:
for table in tables:
    print(f"=== samples.tpch.{table} ===")
    for col in describe_table("samples", "tpch", table):
        nullable = "NULL" if col.get("nullable", True) else "NOT NULL"
        print(f"  {col['col_name']:25} {col['data_type']:20} {nullable}")
    print()

## 3. Hacer una pregunta al agente

In [ ]:
import time
from app.agent.graph import run_agent

def ask(question: str, session_id: str = "notebook-demo"):
    """Helper para invocar el agente y mostrar el resultado formateado."""
    print(f"Q: {question}\n")
    t0 = time.time()
    r = run_agent(question=question, session_id=session_id)
    elapsed = time.time() - t0
    print(f"  intent:        {r.get('intent')}")
    print(f"  schema_source: {r.get('schema_source')}")
    print(f"  retry_count:   {r.get('retry_count')}")
    print(f"  latency:       {elapsed:.1f}s (wall) / {r.get('latency_ms', 0):.0f}ms (agent)")
    if r.get("error"):
        print(f"  error: {r['error']}")
    if r.get("sql_query"):
        print(f"\n  SQL:\n  {chr(10).join('  ' + line for line in r['sql_query'].split(chr(10)))}")
    print(f"\n  answer: {r.get('answer', '(no answer)')}")
    if r.get("results"):
        print(f"\n  results ({len(r['results'])} rows):")
        for row in r["results"][:5]:
            print(f"    {row}")
        if len(r["results"]) > 5:
            print(f"    ... and {len(r['results']) - 5} more")
    print("\n" + "-" * 70)
    return r

ask("How many customers are in UNITED STATES?", session_id="notebook-q1")

## 4. Multi-turn: la 2da pregunta referencia la 1ra

Notar que usamos el MISMO `session_id` (`notebook-q1`) asi el agente tiene contexto de la respuesta anterior. La palabra "those customers" se resuelve como el filtro de USA del turno 1.

In [ ]:
ask("And how many orders did those customers place in 1995?", session_id="notebook-q1")

## 5. Pregunta con revenue (formula TPC-H = extendedprice * (1 - discount))

In [ ]:
ask("Top 3 nations by total order revenue in 1995", session_id="notebook-q3")

## 6. Inspecionar la memoria de sesion

Los turns se guardan en SQLite (`data/db/sessions.db`) por session_id.

In [ ]:
from app.core.memory import get_history

history = get_history("notebook-q1", limit=10)
print(f"Turns guardados para notebook-q1: {len(history)}\n")
for h in history:
    role = h["role"]
    content = h["content"][:80] + ("..." if len(h["content"]) > 80 else "")
    print(f"  [{role}] {content}")
    if h.get("sql"):
        print(f"    SQL: {h['sql'][:80]}...")
    if h.get("error"):
        print(f"    error: {h['error']}")
    print()

## Para correr el eval completo

Desde la raiz del proyecto, en una terminal:

```bash
python scripts/run_eval.py
```

Toma ~15 min para 30 preguntas. Output va a `data/eval/eval_results.json` con metricas por categoria.